In [2]:
import pandas as pd
import json
import time
import re
from tqdm import tqdm
from google import genai
import os

In [ ]:
# Initialize Gemini Client
client = genai.Client(api_key=os.getenv("GEMINI_KEY"))
df = pd.read_csv("../data/ner/filtered_for_NER.csv")

In [ ]:
def safe_json_parse(text):
    # Remove markdown code fences if present
    text = re.sub(r"```json|```", "", text).strip()
    
    # Convert cleaned string to Python dictionary/list
    return json.loads(text)

# Batch Entity Extraction Function
def extract_entities_batch(text_batch):
    """ 
    This function sends a batch of Arabic sentences to Gemini and 
    asks it to extract financial entities in structured JSON format.
    """    
    
    # Construct prompt with clear instructions
    prompt = f"""
You are a financial NER system.

Extract entities from each Arabic sentence.

Entities:
- AMOUNT
- CURRENCY
- RECIPIENT
- BILL_TYPE

Return ONLY a JSON list.
Each item must follow this format:

{{
  "text": "...",
  "amount": "... or null",
  "currency": "... or null",
  "recipient": "... or null",
  "bill_type": "... or null"
}}

Sentences:
{json.dumps(text_batch, ensure_ascii=False)}
"""
    # Send request to Gemini model
    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=prompt
    )
    # Return raw model output (should be JSON formatted string)
    return response.text

In [ ]:
# batch_size = Number of sentences sent to Gemini in one request.
batch_size = 10
results = []

for i in tqdm(range(0, len(df), batch_size)):
    # Extract batch of texts
    batch_texts = df["text"].iloc[i:i+batch_size].tolist()
    
    # Extract corresponding intents (we already have them labeled)
    batch_intents = df["intent"].iloc[i:i+batch_size].tolist()

    try:
        # Send batch to Gemini for entity extraction
        output = extract_entities_batch(batch_texts)
        
        # Parse model JSON response safely
        parsed = safe_json_parse(output)

        for item, intent in zip(parsed, batch_intents):
            item["intent"] = intent
            
        # Add parsed results to the global results list
        results.extend(parsed)

        pd.DataFrame(results).to_csv(
            "../data/ner/extract_entity.csv",
            index=False
        )

        # Small delay to avoid hitting API rate limits
        time.sleep(1)
        
    # If a batch fails, print the error and continue processing next batches.
    except Exception as e:
        print(f"Error in batch {i}: {e}")

100%|██████████| 3/3 [00:30<00:00, 10.12s/it]
